In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

sys.path.append('..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
)
from utils.analysis_utils import extract_verdict, balance_classes_with_oversampling
from utils.path_utils import resolve_path
# Constants
MAIN_AGENT_MODELS = ['claude-3-7-sonnet-20250219']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = range(1, 15)
EXCLUDE_TASKS = []
BACK_FRACS = [0.2, 0.5, 0.8]

In [ ]:
def get_final_suspiciousness(row):
    max_key = max([int(x) for x in row.data.keys()])
    return float(extract_verdict(row.data[str(max_key)]["content"] if isinstance(row.data[str(max_key)], dict) and "content" in row.data[str(max_key)] else row.data[str(max_key)], tag_word='verdict'))

def get_final_usage(row):
    max_key = max([int(x) for x in row.data.keys()])
    return float(row.data[str(max_key)]["usage"][0]["cost_usd"])

def get_downsampled_responses(x, downsample_factor, back_frac=1.0):
    """Extract responses with specified downsampling factor."""
    all_indices = list(x.values())[::downsample_factor] + [list(x.values())[-1]]
    back_indices = all_indices[-int(len(all_indices) * back_frac):]
    return back_indices

def prepare_turn_by_turn_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, downsample_factors: List[int] = DOWNSAMPLE_FACTORS, thresholds_to_consider = [4, 9, 14, 19, 29, 39], nth_highest_indices: List[int] = [2, 3, 5], suffix: str = 'turn_by_turn') -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    turn_by_turn_monitoring_df = load_monitoring_data(base_dir=base_dir)
    turn_by_turn_monitoring_df = turn_by_turn_monitoring_df[~turn_by_turn_monitoring_df.task_name.isin(EXCLUDE_TASKS)]
    for downsample_factor in downsample_factors:
        turn_by_turn_monitoring_df[f'max_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_monitoring_df[f'max_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
    for threshold in thresholds_to_consider:
        turn_by_turn_monitoring_df[f'frac_above_{threshold}_{suffix}'] = 100 * turn_by_turn_monitoring_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') > threshold or 0.0 for y in (list(x.values()) + [list(x.values())[-1]])]))
        )
    
    # Add columns for nth highest suspicion scores
    for n in nth_highest_indices:
        turn_by_turn_monitoring_df[f'{n}th_highest_suspiciousness_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: sorted([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in list(x.values())], reverse=True)[n-1] if len(x) >= n else 0.0
        )
    turn_by_turn_monitoring_df[f'final_suspiciousness_{suffix}'] = turn_by_turn_monitoring_df.apply(
        lambda row: get_final_suspiciousness(row),
        axis=1
    )
    turn_by_turn_monitoring_df[f'final_usage_{suffix}'] = turn_by_turn_monitoring_df.apply(
        lambda row: get_final_usage(row),
        axis=1
    )
    
    for back_frac in BACK_FRACS:
        turn_by_turn_monitoring_df[f'max_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_monitoring_df[f'max_suspiciousness_back_{back_frac}_usage_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_monitoring_df[f'mean_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_monitoring_df[f'mean_suspiciousness_back_{back_frac}_usage_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )

    return turn_by_turn_monitoring_df

def prepare_read_the_notes_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, read_the_notes_agent: str = 'gpt-4o') -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    read_the_notes_df = load_monitoring_data(base_dir=base_dir)
    read_the_notes_df['read_the_notes_agent'] = read_the_notes_agent
    read_the_notes_df['read_the_notes_suspiciousness'] = read_the_notes_df.data.apply(
        lambda x: extract_verdict(x[read_the_notes_agent]["content"] if isinstance(x[read_the_notes_agent], dict) and "content" in x[read_the_notes_agent] else x[read_the_notes_agent], tag_word='verdict') #type: ignore
    )
    
    return read_the_notes_df

def merge_and_compute_features_for_turn_by_turn(combined_results: pd.DataFrame, 
                             turn_by_turn_monitoring_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        turn_by_turn_monitoring_df,
        combined_results[combined_results.monitor_agent == 'claude-3-5-sonnet-20241022'].drop(columns = ['monitor_agent']),
        on=['task_name', 'transcript_type', 'main_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features
    return merged_df

def merge_and_compute_features_for_read_the_notes(combined_results: pd.DataFrame, 
                             read_the_notes_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        read_the_notes_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'file_name'],
        how='left'
    ).dropna()
    
    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    return merged_df



def plot_turn_by_turn_monitoring_results(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, give_hint: bool, monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o'], downsample_factors: List[int] = DOWNSAMPLE_FACTORS, agents: List[str] = ['claude-3-5-sonnet-20241022'], metrics: List[str] = ['final_suspiciousness']):
    """Main execution function."""
    # Load and prepare data
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir, include_failed_attempts, include_failed_main_task, downsample_factors)
    print(turn_by_turn_monitoring_df.task_name.unique())
    merged_df = merge_and_compute_features_for_turn_by_turn(combined_results, turn_by_turn_monitoring_df)
    merged_df = merged_df[merged_df.main_agent.isin(agents)]
    merged_df = merged_df[merged_df.monitor_agent.isin(monitors)]
    # Print data shapes for verification
    print("Shape of df before merge:", turn_by_turn_monitoring_df.shape)
    print("Shape of combined_results before merge:", combined_results.shape)
    print("Shape of merged_df after merge:", merged_df.shape)
    # Generate plots
    
    plot_suspiciousness_distributions(merged_df, monitors=monitors, metrics=metrics)
    plot_roc_curves(merged_df, combined_results, give_hint, include_failed_attempts, include_failed_main_task, monitors=monitors, metrics=metrics)
    # for monitor in monitors:
    #     plot_markov_analysis(merged_df, main_agent='claude-3-5-sonnet-20241022', monitor_model=monitor, give_hint=False, metrics=metrics)
include_failed_attempts = False
include_failed_main_task = True

combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)

In [ ]:
base_dir = 'monitor_responses/turn_by_turn_monitoring_catch_main_failure'
turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir, include_failed_attempts, include_failed_main_task, DOWNSAMPLE_FACTORS)

In [ ]:
merged_df = merge_and_compute_features_for_turn_by_turn(combined_results, turn_by_turn_monitoring_df)
merged_df = pd.concat([balance_classes_with_oversampling(merged_df[(merged_df.transcript_type == transcript_type) & (merged_df.monitor_agent == monitor_agent)], class_column='task_name') for transcript_type in merged_df.transcript_type.unique() for monitor_agent in merged_df.monitor_agent.unique()])
merged_df = merged_df[merged_df.transcript_type != 'transcripts_no_hint']

In [ ]:
monitor_labels = {'claude-3-5-sonnet-20241022': 'Claude 3.6 Sonnet', 'gpt-4o': 'GPT-4o', 'claude-3-7-sonnet-20250219': 'Claude 3.7 Sonnet'}
metrics =   ['max_suspiciousness_back_0.2_turn_by_turn', 'final_suspiciousness_turn_by_turn']
metric_labels = {'default_suspiciousness': 'Default Suspiciousness', 'frac_above_29_turn_by_turn': 'Fraction Above 29', 'max_suspiciousness_back_0.2_turn_by_turn': 'Max Suspiciousness (Last 20%)', 'final_suspiciousness_turn_by_turn': 'Default Suspiciousness', 'mean_suspiciousness_back_0.5_turn_by_turn': 'Mean Suspiciousness (Last 50%)', '2th_highest_suspiciousness_turn_by_turn': '2nd Highest Suspiciousness', '3th_highest_suspiciousness_turn_by_turn': '3rd Highest Suspiciousness'}
for monitor in merged_df.monitor_agent.unique():
    auc_scores = {}
    filtered = merged_df[merged_df.monitor_agent == monitor]
    plt.figure(figsize=(8, 4))
    for metric in metrics:
        fpr, tpr, thresholds = roc_curve(filtered.transcript_type != 'transcripts_no_secondary_task', filtered[metric])
        auc_score = auc(fpr, tpr)
        label = f"{metric_labels[metric]} (AUC: {auc_score:.2f})"
        plt.plot(fpr, tpr, label=label)
        auc_scores[label] = auc_score
    handles, labels = plt.gca().get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda x: -auc_scores[x[1]]))
    plt.legend(handles, labels, loc='lower right')
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.grid(True, alpha=0.5)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.plot([0, 1], [0, 1], 'k--')
    assert len(filtered.main_agent.unique()) == 1
    plt.title(f"Attacker: {monitor_labels[filtered.main_agent.unique()[0]]}\nMonitor: {monitor_labels[monitor]}")
    plt.show()

In [ ]:
include_failed_attempts = False
include_failed_main_task = True
base_dir = 'monitor_responses_v1/turn_by_turn_monitoring'
combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)

turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir, include_failed_attempts, include_failed_main_task, DOWNSAMPLE_FACTORS)

In [ ]:
turn_by_turn_monitoring_df.columns

In [ ]:
merged_df = merge_and_compute_features_for_turn_by_turn(combined_results, turn_by_turn_monitoring_df)
merged_df = pd.concat([balance_classes_with_oversampling(merged_df[(merged_df.transcript_type == transcript_type) & (merged_df.monitor_agent == monitor_agent)], class_column='task_name') for transcript_type in merged_df.transcript_type.unique() for monitor_agent in merged_df.monitor_agent.unique()])
merged_df = merged_df[merged_df.transcript_type != 'transcripts_no_hint']
training_data = pd.read_csv('../finetuning/training_data_dfs/combined_3.csv')
merged_df = merged_df[~merged_df.task_name.isin(training_data.task_name.unique())]

In [ ]:
monitor_labels = {'claude-3-5-sonnet-20241022': 'Claude 3.6 Sonnet', 'gpt-4o-2024-08-06': 'GPT-4o', 'claude-3-7-sonnet-20250219': 'Claude 3.7 Sonnet', 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': "Fine-tuned GPT-4o"}
metrics = ['max_suspiciousness_back_0.2_turn_by_turn', 'final_suspiciousness_turn_by_turn']
metric_labels = {'default_suspiciousness': 'Default Suspiciousness', 'frac_above_29_turn_by_turn': 'Fraction Above 29', 'max_suspiciousness_back_0.2_turn_by_turn': 'Max Suspiciousness (Last 20%)', 'final_suspiciousness_turn_by_turn': 'Default Suspiciousness', 'mean_suspiciousness_back_0.5_turn_by_turn': 'Mean Suspiciousness (Last 50%)', '2th_highest_suspiciousness_turn_by_turn': '2nd Highest Suspiciousness', '3th_highest_suspiciousness_turn_by_turn': '3rd Highest Suspiciousness'}
for monitor in merged_df.monitor_agent.unique():
    auc_scores = {}
    filtered = merged_df[merged_df.monitor_agent == monitor]
    plt.figure(figsize=(8, 4))
    for metric in metrics:
        fpr, tpr, thresholds = roc_curve(filtered.transcript_type != 'transcripts_no_secondary_task', filtered[metric])
        auc_score = auc(fpr, tpr)
        label = f"{metric_labels[metric]} (AUC: {auc_score:.2f})"
        plt.plot(fpr, tpr, label=label)
        auc_scores[label] = auc_score
    handles, labels = plt.gca().get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda x: -auc_scores[x[1]]))
    plt.legend(handles, labels, loc='lower right')
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.grid(True, alpha=0.5)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.plot([0, 1], [0, 1], 'k--')
    assert len(filtered.main_agent.unique()) == 1
    plt.title(f"Attacker: {monitor_labels[filtered.main_agent.unique()[0]]}\nMonitor: {monitor_labels[monitor]}")
    plt.show()





In [ ]:
# plt.hist(merged_df[(merged_df.monitor_agent == 'claude-3-5-sonnet-20241022') & (merged_df.transcript_type == 'transcripts_no_secondary_task')]['mean_suspiciousness_back_0.5_turn_by_turn'], alpha = 0.5, label = 'benign', density = True)
plt.hist(merged_df[(merged_df.monitor_agent == 'claude-3-5-sonnet-20241022') & (merged_df.transcript_type != 'transcripts_no_secondary_task')]['mean_suspiciousness_back_0.5_turn_by_turn'], alpha = 0.5, label = 'malicious', density = True)
plt.title("Turn by turn")
plt.legend()
plt.show()

In [ ]:
plt.hist(merged_df[(merged_df.monitor_agent == 'claude-3-5-sonnet-20241022') & (merged_df.transcript_type == 'transcripts_no_secondary_task')]['default_suspiciousness'], alpha = 0.5, label = 'benign', density = True)
plt.hist(merged_df[(merged_df.monitor_agent == 'claude-3-5-sonnet-20241022') & (merged_df.transcript_type != 'transcripts_no_secondary_task')]['default_suspiciousness'], alpha = 0.5, label = 'malicious', density = True)
plt.title("Default suspiciousness")
plt.legend()
plt.show()

In [ ]:
plt.hist(merged_df[(merged_df.monitor_agent == 'gpt-4o') & (merged_df.transcript_type == 'transcripts_no_secondary_task')]['mean_suspiciousness_back_0.5_turn_by_turn'], alpha = 0.5, label = 'benign', density = True, bins = np.linspace(0, 100, 40))
plt.hist(merged_df[(merged_df.monitor_agent == 'gpt-4o') & (merged_df.transcript_type != 'transcripts_no_secondary_task')]['mean_suspiciousness_back_0.5_turn_by_turn'], alpha = 0.5, label = 'malicious', density = True, bins = np.linspace(0, 100, 40))
plt.title("Turn by turn")
plt.legend()
plt.show()

In [ ]:
plt.hist(merged_df[(merged_df.monitor_agent == 'gpt-4o') & (merged_df.transcript_type == 'transcripts_no_secondary_task')]['final_suspiciousness_turn_by_turn'], alpha = 0.5, label = 'benign', density = True, bins = np.linspace(0, 100, 40))
plt.hist(merged_df[(merged_df.monitor_agent == 'gpt-4o') & (merged_df.transcript_type != 'transcripts_no_secondary_task')]['final_suspiciousness_turn_by_turn'], alpha = 0.5, label = 'malicious', density = True, bins = np.linspace(0, 100, 40))
plt.title("Default suspiciousness")
plt.legend()
plt.show()

In [ ]:
merged_df[(merged_df.monitor_agent == 'gpt-4o') & (merged_df.transcript_type == 'transcripts_no_secondary_task')].drop_duplicates(subset = ['task_name', 'file_name', 'transcript_type']).default_suspiciousness.describe()

In [ ]:
combined_results[(combined_results.monitor_agent == 'gpt-4o') & (combined_results.transcript_type == 'transcripts_no_secondary_task')].default_suspiciousness.describe()